In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier

# === Load Data ===
train = pd.read_csv("Train_Data.csv")
test = pd.read_csv("Test_Data.csv")
train = train.dropna(subset=['age_group'])

# === Initial Prep ===
X = train.drop(columns=['age_group', 'SEQN'])
y = train['age_group'].map({'Adult': 0, 'Senior': 1})
X_test = test.drop(columns=['SEQN'])

# === Advanced Feature Engineering ===
for df in [X, X_test]:
    df['GLU_INS_ratio'] = df['LBXGLU'] / (df['LBXIN'] + 1e-5)
    df['BMI_GLU_product'] = df['BMXBMI'] * df['LBXGLU']
    df['GLT_GLU_ratio'] = df['LBXGLT'] / (df['LBXGLU'] + 1e-5)
    df['activity_bmi'] = df['PAQ605'] * df['BMXBMI']
    df['is_obese'] = (df['BMXBMI'] > 30).astype(int)
    df['is_prediabetic'] = (df['LBXGLU'] > 100).astype(int)
    df['is_diabetic'] = (df['LBXGLU'] > 126).astype(int)
    df['INS_BMI_ratio'] = df['LBXIN'] / (df['BMXBMI'] + 1e-5)
    df['GLU_squared'] = df['LBXGLU'] ** 2
    df['BMI_log'] = np.log(df['BMXBMI'] + 1)
    df['activity_bin'] = pd.cut(df['PAQ605'], bins=[-1, 1, 3, 7], labels=[0, 1, 2])
    df['activity_bin'] = df['activity_bin'].cat.add_categories([3]).fillna(3).astype(int)
    df['is_male'] = (df['RIAGENDR'] == 1).astype(int)
    df['insulin_z'] = (df['LBXIN'] - df['LBXIN'].mean()) / (df['LBXIN'].std() + 1e-5)
    

# === Preprocessing ===
prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_prep = prep.fit_transform(X)
X_test_prep = prep.transform(X_test)

# === Stratified K-Fold Setup ===
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
val_probs_all = np.zeros(len(y))
test_probs_all = np.zeros(len(X_test_prep))

# === Define Stacking Ensemble ===
def get_stacked_model():
    base_models = [
        ('xgb', XGBClassifier(n_estimators=200, learning_rate=0.07, random_state=42, use_label_encoder=False, eval_metric='logloss')),
        ('cat', CatBoostClassifier(verbose=0, iterations=300, learning_rate=0.07, random_seed=42)),
        ('lgb', LGBMClassifier(n_estimators=200, learning_rate=0.07, random_state=42))
    ]
    meta_model = LogisticRegression()
    return StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=3, n_jobs=-1, passthrough=True)

# === Train and Evaluate CV Ensemble ===
for train_idx, val_idx in kf.split(X_prep, y):
    X_train, X_val = X_prep[train_idx], X_prep[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = get_stacked_model()
    model.fit(X_train, y_train)

    val_probs_all[val_idx] = model.predict_proba(X_val)[:, 1]
    test_probs_all += model.predict_proba(X_test_prep)[:, 1] / kf.n_splits

# === Threshold Tuning ===
thresholds = np.linspace(0.1, 0.9, 81)
f1s = [f1_score(y, val_probs_all > t) for t in thresholds]
best_threshold = thresholds[np.argmax(f1s)]
print(f"✅ Best threshold: {best_threshold:.2f} | CV F1: {max(f1s):.4f}")

# === Predict Final Test ===
final_preds = (test_probs_all > best_threshold).astype(int)

# === Save Submission File ===
submission = pd.DataFrame({'age_group': final_preds})
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv generated. Go upload and crush it!")


C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier wa

✅ Best threshold: 0.19 | CV F1: 0.4382
✅ submission.csv generated. Go upload and crush it!
